

<center><font face="Times New Roman" size=10><b><i>NLP-Driven Personalized Movie Recommendation Engine</b></i></center></font>


<center><p float="center">
  <img src="https://miro.medium.com/v2/resize:fit:1400/1*qR08Jxq0IHdvFtBsUhCe3Q.jpeg" width="920"/>
</p></center>

# **Problem Statement**

## **Business Context**

As a growing player in the Over-The-Top (OTT) streaming market, **Streamora** has invested in a movie recommendation system meant to help subscribers discover content quickly instead of endlessly browsing the catalog. Recommendations currently lean on basic attributes of a title — its name, its genre tags, and a short synopsis — to decide what to show a viewer next.

Platform analytics, however, point to a clear problem: a large share of users **ignore the recommended titles** and instead search manually for something to watch. This is a red flag for a subscription business for several reasons:

- **Engagement risk** — Users who can't find something to watch quickly are more likely to disengage from a session, or from the platform altogether.
- **Retention risk** — A recommendation engine that doesn't feel "personal" erodes the perceived value of the subscription, increasing churn risk.
- **Content discovery inefficiency** — Streamora has a large content library; if the engine cannot connect users to relevant titles, long-tail content essentially goes unwatched, wasting licensing spend.
- **Competitive pressure** — Rival OTT platforms increasingly use richer, semantically-aware recommendation techniques (as opposed to simple keyword or genre matching), raising the bar for what users expect.

Understanding *why* the current recommendations under-deliver — and quantifying how much better a more sophisticated, language-aware approach could perform — is therefore directly tied to user satisfaction, watch-time, and ultimately subscriber retention and revenue for Streamora.

##  **Objective**

The current recommendation approach is not resonating with users, and Streamora needs an evidence-based way to decide what to build next.

**Objective of this analysis:**

1. **Diagnose** how well the existing ("past") recommendation logic has been performing, using 11 months of historical viewing/recommendation-outcome data.
2. **Design and build alternative, content-based recommendation engines** that use the textual metadata already available for every movie (title, genres, overview) — specifically:
   - A **Word2Vec**-based approach that learns word-level embeddings from the movie corpus itself.
   - A **Sentence Transformer**-based approach that uses a pre-trained transformer model to capture full-sentence semantic meaning.
3. **Evaluate** each candidate approach against the same historical ground truth (what the user actually watched next) so that performance is directly comparable to the current system.
4. **Recommend** the best-performing approach for production use, and demonstrate how it would be applied to generate recommendations for a new, unseen user.

**Success criteria:** A new recommendation approach is considered a viable replacement for the current system if it achieves a materially higher "success rate" — the percentage of times the movie a user actually watched next appears in the model's top-N recommended list — than the current ~13.85% baseline.

## **Data Description**

**Movie Data**

* **title**: Name of the movie.
* **genres**: Space-separated list of genres associated with the movie (e.g., *Action Drama*).
* **overview**: Short summary describing the movie plot or storyline.




**Evaluation Data**

* **movie\_1** to **movie\_7**: Individual columns representing the 7 most recently watched movies by the user.
* **date**: Date on which the evaluation or recommendation is recorded.
* **movie\_watch**: The movie the user actually watched after the recommendation.
* **past_success**: Indicates whether the previous model correctly recommended the watched movie (`True`/`False`).


# **Importing Necessary Libraries**

We install specific tested library versions to ensure compatibility and avoid errors during development.


In [ ]:
!pip install \
    numpy==1.26.4 \
    scikit-learn==1.6.1\
    scipy==1.13.1\
    gensim==4.3.3 \
    sentence-transformers==3.4.1 \
    gradio==5.33.0\
    pandas==2.2.2

Note:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

**Interpretation:** This cell imports every library the rest of the notebook depends on:
- `pandas` / `numpy` — data loading, manipulation, and numerical operations.
- `gensim.models.Word2Vec` — to train our own word-embedding model on the movie corpus.
- `sentence_transformers.SentenceTransformer` — to load a pre-trained transformer model for sentence-level embeddings.
- `sklearn.metrics.pairwise.cosine_similarity` — to measure similarity between embedding vectors, which is the core mechanic behind both recommenders.
- `warnings` is silenced so that non-critical dependency warnings don't clutter the notebook output.

In [ ]:
import random

# Set seeds for reproducibility
seed = 42
np.random.seed(seed)
random.seed(seed)

**Interpretation:** A fixed random seed (`42`) is applied to both `numpy` and Python's built-in `random` module. This ensures that any step involving randomness (e.g., Word2Vec's internal initialization) produces the **same result every time the notebook is re-run**, which is essential for reproducible comparisons between models.

# **Loading the Data**

We’ll use the Pandas library to load the data. Pandas makes it easy to work with tables of data and take a quick look at what’s inside.

Let’s load the dataset and see what it looks like.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Interpretation:** Since this notebook is designed to run on Google Colab, this cell mounts the user's Google Drive so that the dataset files stored there become accessible to the notebook's file system (under `/content/drive/...`).

In [ ]:
movie_data = pd.read_csv('/content/drive/MyDrive/Dataset/GenAIDataset/movie_dataset_title.csv')


**Interpretation:** The movie catalog (`movie_dataset_title.csv`) is read into the `movie_data` DataFrame. This is the master dataset of movies — with `title`, `genres`, and `overview` — that both recommendation approaches will be built on top of.

In [ ]:
evaluation_data=pd.read_csv('/content/drive/MyDrive/Dataset/GenAIDataset/Evaluation_Data.csv')

**Interpretation:** The historical evaluation log (`Evaluation_Data.csv`) is read into `evaluation_data`. This dataset captures, for 2,000 past recommendation instances, the 7 most recently watched movies for a user, the movie they watched next, and whether the *old* model's recommendation was a hit. This is the ground truth we will use to score every candidate model.

# **Data Overview**

## Movie Data

In [ ]:
movie_data.shape

**Interpretation:** The dataset consists of 4803 rows and 3 columns

In [ ]:
movie_data.head(10)


**Interpretation:** A quick visual check of the first 10 rows confirms the structure of the data: each row is one movie, with a `title`, a space/pipe-separated `genres` string, and a free-text `overview`. This gives us confidence the columns loaded correctly and previews the kind of raw text we'll need to clean and embed.

In [ ]:
movie_data.info()


**Interpretation:** The dataset contains 4,803 movie records with 3 key features:

- title: Movie name — available for all entries.
- overview: Missing in 3 records.
- genres: Missing in 28 records.

In [ ]:
movie_data.dropna(inplace=True)

**Interpretation:** Rows with missing `overview` or `genres` values are dropped. Since our recommendation approaches rely entirely on the *combined text* of a movie (title + genres + overview), a movie with a missing field cannot be reliably embedded, so removing these ~31 incomplete rows protects the quality of the downstream embeddings.

In [ ]:
movie_data.reset_index(drop=True, inplace=True)

**Interpretation:** After dropping rows, the DataFrame index is reset (`0, 1, 2, ...`) so there are no gaps left behind by the removed rows. This keeps positional indexing (e.g., `.iloc[]`) consistent and safe to use later in the notebook.

## Evaluation Data

In [ ]:
evaluation_data.shape

**Interpretation:** The dataset consists of 2000 rows and 10 columns

In [ ]:
evaluation_data.head(10)

**Interpretation:** Previewing the first 10 rows of `evaluation_data` confirms the expected structure: seven "recently watched" movie columns (`movie_1`...`movie_7`), the `movie_watch` target column, a `date`, and the `past_success` flag from the legacy model. This is effectively our test set for every model we build.

In [ ]:
evaluation_data.info()

**Interpretation:** The dataset contains **2000 rows** and includes the following columns:

* **`movie_1` to `movie_7`**: These 7 columns represent recently watched movies by a user. Each column contains movie names as **object** datatype.
* **`movie_watch`**: An **object** datatype column indicating the movie currently being watched.
* **`past_success`**: A **boolean** column indicating whether past recommendations were successful.
* **`date`**: An **object** datatype column representing the date information.

There are **no missing values** in the dataset.


# **EDA**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert the 'date' column to datetime objects
# This is necessary to extract month and year information easily.
evaluation_data['date'] = pd.to_datetime(evaluation_data['date'])

# Extract month and year from the 'date' column
# We combine year and month to group data by calendar month across years.
evaluation_data['month_year'] = evaluation_data['date'].dt.to_period('M')

# Group by month and count the occurrences of each 'Past success' value
# This gives us the total count of True and False successes for each month.
monthly_success_counts = evaluation_data.groupby('month_year')['past_success'].value_counts().unstack(fill_value=0)

# Calculate the percentage of 'True' success for each month
# We divide the count of True successes by the total count (True + False) for that month and multiply by 100.
monthly_success_counts['Success Percentage'] = (monthly_success_counts[True] / (monthly_success_counts[True] + monthly_success_counts[False])) * 100

# Sort the data by month and year
# This ensures the plot displays the months in chronological order.
monthly_success_counts = monthly_success_counts.sort_index()

# Create the plot
# We use seaborn for a visually appealing bar plot.
plt.figure(figsize=(12, 6)) # Set the size of the figure for better readability
sns.barplot(x=monthly_success_counts.index.astype(str), y='Success Percentage', data=monthly_success_counts) # Create a bar plot with month_year on x-axis and Success Percentage on y-axis. Use a colormap 'viridis'.

# Add labels and title to the plot
plt.xlabel('Month-Year') # Label for the x-axis
plt.ylabel('Success Rate (%)') # Label for the y-axis
plt.title('Monthly Success Rate of Past Model Predictions') # Title of the plot
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability, especially if there are many months
plt.tight_layout() # Adjust layout to prevent labels from overlapping

# Display the plot
plt.show()

**Observation:**

**Interpretation:** The success rate of past model recommendations varies **between 10% and 19.4%** across the months.

* The **highest success rate** was observed in **April 2024 (\~19.4%)**, indicating peak performance of the past recommendation model.
* The **lowest success rate** was recorded in **September 2024 (\~10%)**, showing a drop in effectiveness.


In [ ]:
# Calculate the overall success percentage
overall_success_percentage = (evaluation_data['past_success'].sum() / len(evaluation_data)) * 100

# Create a figure and an axes
fig, ax = plt.subplots(figsize=(6, 4))

# Create a bar plot for overall success
ax.bar(['Overall Success'], [overall_success_percentage], color='skyblue')

# Add the percentage value on top of the bar
ax.text('Overall Success', overall_success_percentage + 1, f'{overall_success_percentage:.2f}%', ha='center')

# Add labels and title
ax.set_ylabel('Success Rate (%)')
ax.set_title('Overall Success Rate of Past Model Predictions')
ax.set_ylim(0, 100) # Set y-axis limit to 0-100%

# Display the plot
plt.show()

**Observation**

**Interpretation:** The past model achieved a 13.85% success rate in recommending movies that were actually watched.


# **Model Building**

In this project, we’ll experiment with **two different approaches** to generate vector representations of movie metadata:

1. **Word2Vec-based model** – which learns word-level embeddings from tokenized text inputs.
2. **Sentence Transformer-based model** – which captures semantic meaning at the sentence level using pre-trained transformer architectures.

These embeddings will serve as the foundation for calculating movie-to-movie similarity and powering our recommendation system.

Let’s begin by preparing the data and training the first embedding model.


## Word2Vec

**Word2Vec** is a technique that converts words into vector representations based on their contextual relationships in text. These word embeddings capture semantic meaning and can be averaged to represent longer pieces of text like movie descriptions.


**Process of Building a Word2Vec-Based Recommendation System**

1. **Text Preparation**
   Combine the relevant textual fields—**title**, **genres**, and **overview**—into a single string for each movie. Preprocess this combined text by tokenizing, lowercasing, and removing stopwords or punctuation to prepare it for model training.

2. **Model Training**
   Train a Word2Vec model on the preprocessed tokens. This model learns vector representations for each word based on their co-occurrence patterns in the corpus.

3. **Average Vector Calculation**
   Define a function to compute the average Word2Vec embedding for any given text. This is done by averaging the vectors of all valid tokens present in the trained vocabulary.

4. **Embedding Computation**
   For each movie, compute a **single embedding** by taking the average of the word vectors from the combined text (title + genres + overview).

5. **Generating Recommendations**
   To recommend movies:

   * Take the list of recently watched movies by a user.
   * Compute the average embedding of these movies using their content embeddings.
   * Calculate cosine similarity between this average vector and all other movie embeddings.
   * Return the top-ranked movies that are most similar to the user's recent viewing history.


### Model Building

#### <font size=4>**Step 1: Text Preperation**

Combines important text fields (like title, genre, and description) and cleans them by removing punctuation, stopwords, and converting everything to lowercase. This helps the model focus on the actual content.

In [ ]:
data_word2vec=movie_data.copy()

**Interpretation:** A dedicated copy, `data_word2vec`, is created from `movie_data`. Working on a copy (rather than modifying `movie_data` directly) keeps the original clean dataset reusable for the Sentence Transformer approach later in the notebook, so the two modeling paths don't interfere with each other.

**NOTE:** The `simple_preprocess` function from `gensim.utils` tokenizes the text, lowercases all words, and removes punctuation and very short tokens. It prepares the text for training the Word2Vec model by converting it into a clean list of meaningful words.


In [ ]:
from gensim.utils import simple_preprocess
data_word2vec['text'] = data_word2vec['title'] + ' ' + data_word2vec['genres'].replace('|', ' ', regex=True) + ' ' + data_word2vec['overview']
data_word2vec['tokens'] = data_word2vec['text'].apply(lambda x: simple_preprocess(x))

**Interpretation:** The `title`, `genres`, and `overview` fields are concatenated into a single `text` field per movie, and pipe characters in `genres` are replaced with spaces. `simple_preprocess` then tokenizes this combined text into a clean list of lowercase words (`tokens`), stripping punctuation and very short tokens. The result is the exact input format Word2Vec needs for training — one list of tokens per movie.

####<font size=4>**Step 2: Model Training**

Trains the system to understand word meanings based on how often and where words appear together. This helps capture relationships between similar or related terms.

In [ ]:
model_word2vec = Word2Vec(sentences=data_word2vec['tokens'], vector_size=100, window=5, min_count=1, workers=4,seed=42)

**Interpretation:** A Word2Vec model is trained **from scratch** directly on our movie corpus (rather than using a generic pre-trained model), so the resulting word vectors are tuned to movie-specific vocabulary and co-occurrence patterns. With `vector_size=100` and `window=5`, each word learns a 100-dimensional representation based on the words that typically surround it within a 5-word context window, and `min_count=1` ensures no vocabulary (even rare terms) is dropped. `seed=42` keeps training reproducible.

#### <font size=4>**Step 3: Average Vector Calculation**

Averages the embeddings of the individual words to provide a unified representation that captures the overall semantic meaning of the sentence by combining the contributions from all words

- Also makes it easier to compare sentences of different lengths as they're all transformed into vectors of the same dimension.

In [ ]:
def get_avg_word_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)


**Interpretation:** This helper function converts a list of tokens into a single fixed-length vector by averaging the Word2Vec vectors of every token that exists in the trained vocabulary. If none of the tokens are recognized, it safely falls back to a zero-vector rather than raising an error. This averaging step is what turns *word-level* embeddings into a *movie-level* embedding.

#### <font size=4>**Step 4: Embedding Computation**

Converts each word into a fixed-length vector based on its content. These vectors help the system compare how similar different items are.

In [ ]:
data_word2vec['embedded_vector'] = data_word2vec['tokens'].apply(lambda x: get_avg_word_vector(x, model_word2vec))

**Interpretation:** This step applies the averaging function to every movie's token list, producing a single `embedded_vector` per movie in `data_word2vec`. It calculates the average word vector for each list of tokens using the Word2Vec model.


#### <font size=4>**Step 5: Generating Recommedations**


Using the average embedding of recently watched movies captures the user's general taste. Comparing this with other movies using cosine similarity helps find content that is semantically similar. Excluding already-watched items ensures that the recommendations are fresh and relevant.

In [ ]:
def word2vec_recommendations(recently_watched_titles, data, model, n_recommendations=10):

    # Get embeddings for the recently watched movies
    watched_embeddings = []
    for title in recently_watched_titles:
        movie_row = data[data['title'] == title]
        if not movie_row.empty:
            watched_embeddings.append(movie_row.iloc[0]['embedded_vector'])

    if not watched_embeddings:
        return ["Could not find embeddings for recently watched movies."]

    # Compute the average embedding of the recently watched movies
    avg_watched_embedding = np.mean(watched_embeddings, axis=0)

    # Calculate cosine similarity between the average watched embedding and all movie embeddings
    all_embeddings = np.vstack(data['embedded_vector'].values)
    similarities = cosine_similarity([avg_watched_embedding], all_embeddings)[0]

    # Get the indices of the top n most similar movies
    # Exclude movies that were recently watched
    recommended_indices = similarities.argsort()[::-1] # Sort in descending order

    recommended_titles = []
    for idx in recommended_indices:
        movie_title = data.iloc[idx]['title']
        if movie_title not in recently_watched_titles:
            recommended_titles.append(movie_title)
            if len(recommended_titles) >= n_recommendations:
                break

    return recommended_titles

**Interpretation:** This function is the core Word2Vec recommender: it looks up the embeddings for a user's recently watched movies, averages them into a single "taste vector," measures cosine similarity between that vector and every movie's embedding, and returns the top-N most similar titles — excluding anything the user has already watched. This same logic will later be applied to the whole evaluation set.

Let's generate recommendations from the movies we've already watched.

In [ ]:
watched_movies = ['The Avengers', 'Iron Man', 'Man of Steel']
word2vec_recommendations(watched_movies, data_word2vec, model_word2vec)


**Interpretation:** The Word2vec model produced a mix of results. While movies like *Live Free or Die Hard*, *Oblivion*, and *Species* somewhat aligned with superhero themes through action and sci-fi elements, many recommendations such as *When a Stranger Calls*, *The Good Guy*, and *Creepshow 2* did not match the tone or genre, indicating weak alignment with the original superhero movies.




### Evaluation

**NOTE:** We create a list of past movies so it can be easily passed into the recommendation function.

In [ ]:
past_movies = evaluation_data[['movie_1', 'movie_2', 'movie_3', 'movie_4', 'movie_5', 'movie_6', 'movie_7']].values.tolist()

**Interpretation:** The seven `movie_*` columns from `evaluation_data` are converted into a Python list-of-lists, `past_movies` — one sub-list of recently watched movies per historical record. This reshapes the evaluation set into the exact input format the recommendation functions expect.

In [ ]:
word2vec_recommendations_list = []
for watched_movies_list in past_movies:
    # Filter out None values from the watched_movies_list
    valid_watched_movies = [movie for movie in watched_movies_list if movie is not None]
    recommendations = word2vec_recommendations(valid_watched_movies, data_word2vec, model_word2vec, n_recommendations=10)
    word2vec_recommendations_list.append(recommendations)

# Print the first few recommendations from the word2vec model
print("Word2Vec Recommendations for the first row:", word2vec_recommendations_list[0])

**Interpretation:** For every historical user record, the Word2Vec recommender is run on that user's valid watched-movie history (filtering out any `None` placeholders), producing a list of 10 recommended titles per record. These are stored in `word2vec_recommendations_list`, ready to be checked against what the user actually watched.

In [ ]:
evaluation_data['word2vec_match'] = evaluation_data.apply(lambda row: row['movie_watch'] in word2vec_recommendations_list[row.name], axis=1)

result_df = evaluation_data[['date', 'movie_watch', 'word2vec_match']]
result_df

**Interpretation:** For each historical record, this checks whether the movie the user *actually went on to watch* (`movie_watch`) appears anywhere in that record's Word2Vec-generated recommendation list, storing the result as a boolean `word2vec_match`. This is the same "hit/miss" logic used to score the legacy model, which is what makes the comparison apples-to-apples.

In [ ]:
# Calculate the overall success percentage for Word2Vec
overall_word2vec_success_percentage = (evaluation_data['word2vec_match'].sum() / len(evaluation_data)) * 100

# Create a figure and an axes
fig, ax = plt.subplots(figsize=(6, 4))

# Create a bar plot for overall success
ax.bar(['Overall Word2Vec Success'], [overall_word2vec_success_percentage], color='lightgreen')

# Add the percentage value on top of the bar
ax.text('Overall Word2Vec Success', overall_word2vec_success_percentage + 1, f'{overall_word2vec_success_percentage:.2f}%', ha='center')

# Add labels and title
ax.set_ylabel('Success Rate (%)')
ax.set_title('Overall Success Rate of Word2Vec Model Recommendations')
ax.set_ylim(0, 100) # Set y-axis limit to 0-100%

# Display the plot
plt.show()

**Observation:**
- **Interpretation:** The Word2Vec model achieves an average recommendation success rate of 22.7%.


**NOTE:** Monthly success refers to the percentage of watched movies that were present in the recommendation list provided by the model. It indicates how effectively the model's recommendations align with actual user behavior.


In [ ]:
import matplotlib.pyplot as plt

# Convert 'date' column to datetime objects
result_df['date'] = pd.to_datetime(result_df['date'])

# Extract month and year
result_df['month_year'] = result_df['date'].dt.to_period('M')

# Group by month and count total watches and matches
monthly_summary = result_df.groupby('month_year').agg(
    total_watches=('movie_watch', 'count'),
    successful_recommendations=('word2vec_match', lambda x: (x == True).sum())
).reset_index()

# Calculate success rate
monthly_summary['success_rate'] = (monthly_summary['successful_recommendations'] / monthly_summary['total_watches']) * 100

# Sort by month and year
monthly_summary = monthly_summary.sort_values(by='month_year')

# Convert month_year to string for plotting
monthly_summary['month_year_str'] = monthly_summary['month_year'].astype(str)

# Create the bar chart
plt.figure(figsize=(12, 6))
plt.bar(monthly_summary['month_year_str'], monthly_summary['success_rate'], color='skyblue')
plt.xlabel('Month')
plt.ylabel('Success Rate (%)')
plt.title('Monthly Recommendation Success Rate (Word2Vec)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Observation**

* **Interpretation:** The **success rate ranges between \~17% and 27%**, indicating moderate performance of the Word2Vec-based recommendation system.
* The **highest success rate** is observed in **Feb 2024 (\~27%)**.
* The **lowest success rate** occurs in **September and November 2024 (\~16–17%)**

**Note:** The observations presented are based on the output obtained after training our model. However, since we're using neural network-based architectures here, the results may vary slightly with multiple executions.

## Sentence Transformer

**Sentence Transformers** are models that generate dense vector representations (embeddings) of sentences or texts. These embeddings capture the semantic meaning of the input text, making them suitable for tasks like similarity comparison and recommendation.

**Process of Building a SentenceTransformer-Based Recommendation System**

1. **Creating a Separate Embedding DataFrame**
   We create a new DataFrame, `data_sentf`, to store the SentenceTransformer-based embeddings independently, ensuring the original movie data remains unmodified.

2. **Model Loading**
   Load a pre-trained SentenceTransformer model such as `'all-MiniLM-L6-v2'`, which offers a good balance of performance and efficiency for semantic similarity tasks.

3. **Generating and Storing Combined Embeddings**
   For each movie, compute individual embeddings for the **title**, **genres**, and **overview** using the loaded SentenceTransformer model. Add these embeddings together to form a single combined embedding that captures the movie’s overall content. Store the final embeddings in `data_sentf` for similarity comparison.

4. **Generating Recommendations**
   To generate recommendations:

   * Take a list of movies recently watched by a user.
   * Compute the average embedding of these movies.
   * Calculate cosine similarity between this average embedding and those of all other movies.
   * Return the top-ranked similar movies, excluding those already watched.



### Model Building

#### <font size=4>**Step 1: Creating a Separate Embedding DataFrame**

We created `data_sentf` to store embeddings independently without modifying the original DataFrame.


In [ ]:
data_sentf=movie_data.copy()

**Interpretation:** A second independent copy, `data_sentf`, is created from the clean `movie_data`, so the Sentence Transformer embeddings can be built and stored without touching the Word2Vec working copy (`data_word2vec`) or the original data.

#### <font size=4>**Step 2: Model Loading**

Using a pre-trained SentenceTransformer model allows us to leverage powerful language understanding without the need to train from scratch. Models like 'all-MiniLM-L6-v2' are optimized for capturing semantic similarity efficiently.

- We use the all-MiniLM-L6-v2 model to generate compact and efficient sentence embeddings suitable for semantic similarity tasks.
- Alternatively, models like **`paraphrase-MpNet-base-v2`** can also be used, though they may require more computational resources.


In [ ]:
model_sentf = SentenceTransformer('all-MiniLM-L6-v2')

**Interpretation:** This loads the pre-trained `all-MiniLM-L6-v2` Sentence Transformer model. Unlike Word2Vec, this model has already learned general-purpose sentence semantics from a very large text corpus, so it can be used directly for inference (no training step is required) while still being lightweight enough to embed thousands of movies quickly.

#### <font size=4>**Step 3: Generating and Storing Combined Embeddings**

Creating embeddings for each text field (title, genres, overview) allows the model to capture different aspects of the content. Adding them together gives a balanced and complete representation. Storing these combined embeddings makes the similarity comparison faster and avoids repeated computation.

In [ ]:

def get_combined_embedding(title, genres, overview, model):
    # Create a combined string
    combined_text = f"{title} {genres} {overview}"
    # Generate embedding for the combined text
    embedding = model.encode(combined_text)
    return embedding

# Apply the function to create embeddings for all movies
data_sentf['embedded_vector'] = data_sentf.apply(
    lambda row: get_combined_embedding(row['title'], row['genres'], row['overview'], model_sentf), axis=1
)

**Interpretation:** For every movie, `title`, `genres`, and `overview` are combined into one string and passed through the Sentence Transformer to produce a single dense embedding capturing the movie's overall semantic content. Unlike the Word2Vec averaging approach, this embedding is generated holistically by the transformer, allowing it to capture context and word order rather than just a bag-of-words average.

#### <font size=4>**Step 4: Generating Recommendations**

Using the average embedding of recently watched movies captures the user's general taste. Comparing this with other movies using cosine similarity helps find content that is semantically similar. Excluding already-watched items ensures that the recommendations are fresh and relevant.

In [ ]:
def recommend_movies_sentf(watched_movies_titles, data, model, n=10):

    watched_movies_embeddings = []
    for title in watched_movies_titles:
        movie = data[data['title'] == title]
        if not movie.empty:
            # Use 'combined_embedding' column here
            watched_movies_embeddings.append(movie.iloc[0]['embedded_vector'])


    if not watched_movies_embeddings:
        return []

    # Calculate the average embedding of watched movies
    avg_embedding = np.mean(watched_movies_embeddings, axis=0)

    # Calculate cosine similarity with all other movies using 'combined_embedding'
    similarities = cosine_similarity([avg_embedding], np.vstack(data['embedded_vector'].values))

    # Get the indices of the most similar movies (excluding the watched ones)
    similar_indices = similarities.argsort()[0][::-1]

    recommended_movies = []
    for index in similar_indices:
        movie_title = data.iloc[index]['title']
        if movie_title not in watched_movies_titles:
            recommended_movies.append(movie_title)
        if len(recommended_movies) == n:
            break

    return recommended_movies


**Interpretation:** This mirrors the Word2Vec recommender but operates on the Sentence Transformer embeddings: it averages the embeddings of a user's recently watched movies into a "taste vector," ranks all movies by cosine similarity to that vector, and returns the top-N most similar titles not already watched.

Let's generate recommendations from the movies we've already watched.

In [ ]:
last_three = ['The Avengers', 'Iron Man', 'Man of Steel']
res=recommend_movies_sentf(last_three,data_sentf,model_sentf,10)
res

**Interpretation:** The Sentence Transformer model delivered strong, relevant recommendations. Titles like *Iron Man 2*, *Avengers: Age of Ultron*, *Captain America: Civil War*, and *Thor* are directly connected to the original superhero universe or share similar themes of action, heroism, and world-building. These suggestions show a much better understanding of genre and viewer intent.

### Evaluation

In [ ]:

sentf_recommendations = []
for watched_movies_list in past_movies:
    # Filter out None values from the watched_movies_list
    valid_watched_movies = [movie for movie in watched_movies_list if movie is not None]
    recommendations = recommend_movies_sentf(valid_watched_movies, data_sentf, model_sentf, n=10)
    sentf_recommendations.append(recommendations)

# Print the first few recommendations from the sentence transformer model
print("Sentence Transformer Recommendations for the first row:", sentf_recommendations[0])

**Interpretation:** The same historical `past_movies` list used for the Word2Vec evaluation is reused here, this time run through the Sentence Transformer recommender, so both models are scored on **identical** historical user histories — a fair, controlled comparison.

In [ ]:

evaluation_data['sentf_match'] = evaluation_data.apply(
    lambda row: row['movie_watch'] in sentf_recommendations[row.name], axis=1
)

result_df = evaluation_data[['date', 'movie_watch', 'sentf_match']]
result_df


**Interpretation:** As with Word2Vec, this checks whether each user's actually-watched movie appears in that record's Sentence Transformer recommendation list, producing the `sentf_match` boolean column used to compute the model's success rate.

In [ ]:

# Calculate the overall success percentage for the Sentence Transformer model
overall_success_percentage_sentf = (evaluation_data['sentf_match'].sum() / len(evaluation_data)) * 100

# Create a figure and an axes
fig, ax = plt.subplots(figsize=(6, 4))

# Create a bar plot for overall success
ax.bar(['Overall Success (Sentence Transformer)'], [overall_success_percentage_sentf], color='lightgreen')

# Add the percentage value on top of the bar
ax.text('Overall Success (Sentence Transformer)', overall_success_percentage_sentf + 1, f'{overall_success_percentage_sentf:.2f}%', ha='center')

# Add labels and title
ax.set_ylabel('Success Rate (%)')
ax.set_title('Overall Success Rate of Sentence Transformer Model Predictions')
ax.set_ylim(0, 100) # Set y-axis limit to 0-100%

# Display the plot
plt.show()


**Observation:**
- **Interpretation:** The Sentence Transformer model achieves an average recommendation success rate of 64.75%.


In [ ]:
# Convert 'date' column to datetime objects
result_df['date'] = pd.to_datetime(result_df['date'])

# Extract month and year
result_df['month_year'] = result_df['date'].dt.to_period('M')

# Group by month and count total watches and matches
monthly_summary_sentf = result_df.groupby('month_year').agg(
    total_watches=('movie_watch', 'count'),
    successful_recommendations=('sentf_match', lambda x: (x == True).sum())
).reset_index()

# Calculate success rate
monthly_summary_sentf['success_rate'] = (monthly_summary_sentf['successful_recommendations'] / monthly_summary_sentf['total_watches']) * 100

# Sort by month and year
monthly_summary_sentf = monthly_summary_sentf.sort_values(by='month_year')

# Convert month_year to string for plotting
monthly_summary_sentf['month_year_str'] = monthly_summary_sentf['month_year'].astype(str)

# Create the bar chart
plt.figure(figsize=(12, 6))
plt.bar(monthly_summary_sentf['month_year_str'], monthly_summary_sentf['success_rate'], color='lightcoral')
plt.xlabel('Month')
plt.ylabel('Success Rate (%)')
plt.title('Monthly Recommendation Success Rate (Sentence Transformer)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


**Observation**

* **Interpretation:** The success rate ranges between **\~62% and 68%**, indicating **strong and consistent performance** of the Sentence Transformer-based recommendation system.
    - This drastic improvement in performance is owing to the fact that Sentence Transformers capture contextual and semantic meaning at the sentence level, unlike Word2Vec which only generates static word-level embeddings.
* The **highest success rate** is observed in **March and August 2024 (\~67–68%)**.
* The **lowest success rate** occurs in **October 2024 (\~63%)**.
* Most months maintain a success rate above **64%**, showing **robust recommendation quality** across the year.




# **Model Comparison**

The Sentence Transformer model significantly **outperforms** both:

* The **Word2Vec-based model** (which ranged between \~17% to 27%)
* The **Past recommendation model** (which had lower success percentages overall of ~13.5%)

This demonstrates that **contextually-rich sentence embeddings from transformers** provide a more accurate and reliable basis for generating recommendations.

# **Conclusion**

### 1. Key Findings

| Recommendation Approach | Overall Success Rate | Nature of Embedding |
|---|---|---|
| **Past (legacy) model** | ~13.85% | Non-semantic / rules-based |
| **Word2Vec (custom-trained)** | ~22.7% | Word-level, averaged, static |
| **Sentence Transformer (`all-MiniLM-L6-v2`)** | **~64.75%** | Sentence-level, contextual, pre-trained |

- Both content-based approaches **comfortably outperformed** the legacy recommendation system, confirming that leveraging movie text (title, genres, overview) adds real predictive signal that the old system was not capturing.
- The **Sentence Transformer approach is the clear winner**, roughly **2.9x** the success rate of Word2Vec and **~4.7x** the success rate of the legacy model.
- Word2Vec's weaker performance is consistent with its design: it averages independent *word*-level vectors, which loses word order and contextual nuance (e.g., it cannot easily tell "man of steel" apart from unrelated action/sci-fi language). The Sentence Transformer instead encodes full phrases contextually, which is why its top-line recommendations (e.g., for *The Avengers*, *Iron Man*, *Man of Steel*) were dominated by genuinely related superhero titles.
- Monthly performance for the Sentence Transformer model was also **more stable** (~62%–68% band) than Word2Vec (~17%–27% band), suggesting more consistent user experience month over month, not just a better average.

### 2. Business Impact for Streamora

- Moving from a ~14% to a ~65% historical hit-rate implies substantially more sessions where a user's next watch is *already* on their recommendation rail — directly addressing the "users bypass recommendations to search manually" problem identified in the business context.
- Better content-to-viewer matching should support two commercial goals simultaneously: **higher engagement/watch-time** (less time spent searching) and better **utilization of the long-tail catalog** (since the model is not limited to a small set of obviously popular titles).
- Because the Sentence Transformer model is pre-trained, it can be deployed **without a lengthy in-house training cycle**, and it can be periodically refreshed simply by re-encoding the (small, text-only) movie catalog — a low operational overhead relative to the performance gain.

### 3. Limitations & Future Work

- **Evaluation scope:** Success is measured as "did the actually-watched movie appear in the top-10 list," which is a reasonable proxy but does not capture ranking quality (e.g., was it recommendation #1 or #10) or how users react to less obviously related recommendations.
- **Content-only signal:** Both models rely solely on text metadata (title, genres, overview). They do not yet incorporate collaborative signals (what *similar users* watched), popularity/recency effects, or explicit user ratings — all of which could push performance further.
- **Cold-start coverage:** Because recommendations depend on embedding similarity to a user's watch history, brand-new users with no viewing history are not addressed by this notebook's approach and would need a separate strategy (e.g., trending/onboarding recommendations).
- **Suggested next steps:** pilot the Sentence Transformer model in an A/B test against the legacy system on live traffic; consider hybridizing it with collaborative filtering signals; and explore stronger transformer variants (e.g., `paraphrase-mpnet-base-v2`) if latency budgets allow, to see if additional accuracy is available.

### 4. Final Verdict

The **Sentence Transformer-based recommendation engine (`all-MiniLM-L6-v2`) is recommended as the model to take forward**, given its clear, consistent, and substantial outperformance of both the legacy system and the custom Word2Vec approach across the full 11-month evaluation window.

## **Generating Recommendations for Unseen Data (Best Model)**

Having identified the **Sentence Transformer model** (`recommend_movies_sentf`, built on `data_sentf` / `model_sentf`) as the best-performing approach, we now demonstrate how it would be used in production: generating recommendations for a **new/unseen watch history** that was not part of the historical evaluation set used above.

Simply replace the `unseen_watched_movies` list below with any user's recently watched titles (as long as the titles exist in the `movie_data` catalog) to generate fresh, personalized recommendations.

In [ ]:
# --- Unseen data inference using the best model (Sentence Transformer) ---

# Example: a new/unseen user's recently watched movies (not used anywhere in training or evaluation above)
unseen_watched_movies = ['The Dark Knight', 'Inception', 'Interstellar']

# Generate top-10 recommendations using the trained Sentence Transformer recommender
unseen_recommendations = recommend_movies_sentf(
    unseen_watched_movies,
    data_sentf,
    model_sentf,
    n=10
)

print(f"Recently watched (unseen user): {unseen_watched_movies}\n")
print("Top-10 recommended movies:")
for rank, movie in enumerate(unseen_recommendations, start=1):
    print(f"{rank}. {movie}")


**Interpretation:** This cell applies the exact same production-ready function (`recommend_movies_sentf`) used throughout the evaluation, but on a **brand-new watch history** that the model has never scored before. It encodes the unseen titles' embeddings (already pre-computed and stored in `data_sentf`), averages them into a taste vector, and returns the 10 most semantically similar unwatched movies — showing exactly how this model would operate in a live recommendation setting for any new user.